# Causal Inference Quickstart
This notebook demonstrates three ideas from the research ticket: **doubly robust
estimation (AIPW)**, **cross-fitting**, and the **target trial** design checklist.
It uses simulated data with a known answer, so it runs anywhere with only
numpy / pandas / scikit-learn.

The goal is to show *what the methods do* and *what questions they answer*,
not to produce clinical results.

In [1]:
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import KFold

rng = np.random.default_rng(42)
PS_CLIP = 0.01   # propensity scores are clipped to [0.01, 0.99] to stop extreme weights

## 1. A confounded observational dataset
Treatment assignment depends on a confounder `X`, and the outcome depends on both
`X` and treatment. The true average treatment effect (ATE) is 1.5. A naive
difference in means mixes the treatment effect with the effect of `X`.

In [2]:
def simulate(n, rng):
    X = rng.normal(0, 1, n)                           # confounder
    A = rng.binomial(1, 1 / (1 + np.exp(-0.8 * X)))   # treatment depends on X
    Y0 = 2.0 * X + rng.normal(0, 1, n)                # outcome under control
    Y1 = 2.0 * X + 1.5 + rng.normal(0, 1, n)          # outcome under treatment
    Y = np.where(A == 1, Y1, Y0)                      # observed outcome
    return pd.DataFrame({"X": X, "A": A, "Y": Y})

true_ate = 1.5
df = simulate(5000, rng)
naive = df[df.A == 1].Y.mean() - df[df.A == 0].Y.mean()
print(f"True ATE:                  {true_ate}")
print(f"Naive difference in means: {naive:.3f}   (bias {naive - true_ate:+.3f}, from confounding by X)")

True ATE:                  1.5
Naive difference in means: 2.971   (bias +1.471, from confounding by X)


## 2. Doubly robust estimation (AIPW)
AIPW combines an outcome model and a propensity model. The ATE estimate is
consistent if **either** model is correctly specified (and the usual assumptions
hold). Here both are correct, so it should land close to the truth.

In [3]:
def aipw_ate(df, outcome_model, propensity_model, clip=PS_CLIP):
    X, A, Y = df[["X"]].values, df["A"].values, df["Y"].values
    ps = np.clip(clone(propensity_model).fit(X, A).predict_proba(X)[:, 1], clip, 1 - clip)
    m1 = clone(outcome_model).fit(X[A == 1], Y[A == 1]).predict(X)   # E[Y | A=1, X]
    m0 = clone(outcome_model).fit(X[A == 0], Y[A == 0]).predict(X)   # E[Y | A=0, X]
    psi = (m1 - m0) + A * (Y - m1) / ps - (1 - A) * (Y - m0) / (1 - ps)
    return psi.mean(), psi.std(ddof=1) / np.sqrt(len(psi))

ate, se = aipw_ate(df, LinearRegression(), LogisticRegression(max_iter=1000))
print(f"AIPW ATE: {ate:.3f} (SE {se:.3f}, 95% CI [{ate - 1.96 * se:.3f}, {ate + 1.96 * se:.3f}])")
print(f"True ATE: {true_ate}")

AIPW ATE: 1.540 (SE 0.031, 95% CI [1.478, 1.601])
True ATE: 1.5


## 3. Cross-fitting
With flexible learners (here gradient boosting), fitting and predicting on the same
rows can overfit the nuisance functions into the estimate. Cross-fitting fits
nuisances on K-1 folds and predicts on the held-out fold. It is the backbone of
double/debiased ML (DML).

Every fit uses `clone(...)` so each fold and treatment arm gets a fresh model.
(sklearn's `fit()` already resets a model, so reusing one object gives the same
numbers, but cloning makes that independent of the learner's internals.)

In [4]:
def crossfit_aipw(df, outcome_model, propensity_model, n_folds=5, clip=PS_CLIP, seed=42):
    X, A, Y = df[["X"]].values, df["A"].values, df["Y"].values
    n = len(df)
    m1_hat, m0_hat, ps_hat = np.zeros(n), np.zeros(n), np.zeros(n)
    for tr, te in KFold(n_splits=n_folds, shuffle=True, random_state=seed).split(X):
        Xtr, Atr, Ytr = X[tr], A[tr], Y[tr]
        ps_hat[te] = clone(propensity_model).fit(Xtr, Atr).predict_proba(X[te])[:, 1]
        m1_hat[te] = clone(outcome_model).fit(Xtr[Atr == 1], Ytr[Atr == 1]).predict(X[te])
        m0_hat[te] = clone(outcome_model).fit(Xtr[Atr == 0], Ytr[Atr == 0]).predict(X[te])
    ps_hat = np.clip(ps_hat, clip, 1 - clip)
    psi = (m1_hat - m0_hat) + A * (Y - m1_hat) / ps_hat - (1 - A) * (Y - m0_hat) / (1 - ps_hat)
    return psi.mean(), psi.std(ddof=1) / np.sqrt(n)

ate_cf, se_cf = crossfit_aipw(df, GradientBoostingRegressor(random_state=0),
                              LogisticRegression(max_iter=1000))
print(f"Cross-fitted AIPW ATE: {ate_cf:.3f} (SE {se_cf:.3f}, "
      f"95% CI [{ate_cf - 1.96 * se_cf:.3f}, {ate_cf + 1.96 * se_cf:.3f}])")
print(f"True ATE: {true_ate}")

Cross-fitted AIPW ATE: 1.516 (SE 0.033, 95% CI [1.451, 1.581])
True ATE: 1.5


## 4. Is one draw enough?
A single estimate can look good by luck. Repeating the simulation 200 times (n = 2,000
each, linear nuisance models for speed) checks the **bias** and the **coverage** of the
95% interval.

In [5]:
R, n_mc = 200, 2000
naive_r, est_r, covered = [], [], []
for r in range(R):
    d = simulate(n_mc, np.random.default_rng(1000 + r))
    naive_r.append(d[d.A == 1].Y.mean() - d[d.A == 0].Y.mean())
    est, se_r = aipw_ate(d, LinearRegression(), LogisticRegression(max_iter=1000))
    est_r.append(est)
    covered.append(abs(est - true_ate) <= 1.96 * se_r)
print(f"{R} replications, true ATE = {true_ate}")
print(f"  naive diff-in-means: mean {np.mean(naive_r):.3f}  (bias {np.mean(naive_r) - true_ate:+.3f})")
print(f"  AIPW:                mean {np.mean(est_r):.3f}  (bias {np.mean(est_r) - true_ate:+.3f}, sd {np.std(est_r):.3f})")
print(f"  AIPW 95% CI coverage: {np.mean(covered):.3f}")

200 replications, true ATE = 1.5
  naive diff-in-means: mean 2.899  (bias +1.399)
  AIPW:                mean 1.502  (bias +0.002, sd 0.046)
  AIPW 95% CI coverage: 0.945


## 5. Federated variant: client-side vs. centralized DR estimation
Three simulated clients with different sizes and covariate distributions (the third is
small and strongly confounded). The same true ATE (1.5) holds in every client.

Three ways to estimate it:
1. **Centralized**: pool all rows (what federation is trying to avoid).
2. **Federated-exact**: nuisance models are fitted from *aggregates only* (OLS sufficient
   statistics, Newton steps for the logistic model), then each client sends only
   `(n, sum of scores, sum of squared scores)`.
3. **Local-only**: each client runs AIPW on its own data, and the server combines the
   per-client estimates by inverse-variance weighting.

In [6]:
def make_client(n, mu, sd, conf, seed):
    r = np.random.default_rng(seed)
    X = r.normal(mu, sd, n)
    A = r.binomial(1, 1 / (1 + np.exp(-conf * X)))
    Y = 2.0 * X + 1.5 * A + r.normal(0, 1, n)         # true ATE = 1.5 in every client
    return X, A, Y

clients = [make_client(2000, 0.0, 1.0, 0.8, 11),
           make_client(1200, 0.5, 1.0, 0.8, 12),
           make_client(300, 1.0, 1.0, 2.5, 13)]       # small and strongly confounded

def design(X):
    return np.column_stack([np.ones(len(X)), X])

def aipw_scores(X, A, Y, e, b1, b0):
    D = design(X)
    m1, m0 = D @ b1, D @ b0
    return (m1 - m0) + A * (Y - m1) / e - (1 - A) * (Y - m0) / (1 - e)

def clipped(p):
    return np.clip(p, PS_CLIP, 1 - PS_CLIP)

# ---- 1. Centralized (rows pooled) ----------------------------------------
Xc = np.concatenate([c[0] for c in clients])
Ac = np.concatenate([c[1] for c in clients])
Yc = np.concatenate([c[2] for c in clients])
e_c = clipped(LogisticRegression(C=1e6, max_iter=1000).fit(Xc.reshape(-1, 1), Ac)
              .predict_proba(Xc.reshape(-1, 1))[:, 1])
b1_c = np.linalg.lstsq(design(Xc[Ac == 1]), Yc[Ac == 1], rcond=None)[0]
b0_c = np.linalg.lstsq(design(Xc[Ac == 0]), Yc[Ac == 0], rcond=None)[0]
s_c = aipw_scores(Xc, Ac, Yc, e_c, b1_c, b0_c)
ate_c, se_c = s_c.mean(), s_c.std(ddof=1) / np.sqrt(len(s_c))

# ---- 2. Federated-exact: only aggregates leave each client ----------------
# Outcome models: each client sends X'X and X'y per arm; the server solves.
def fed_ols(arm):
    G = sum(design(X[A == arm]).T @ design(X[A == arm]) for X, A, Y in clients)
    h = sum(design(X[A == arm]).T @ Y[A == arm] for X, A, Y in clients)
    return np.linalg.solve(G, h)

b1_f, b0_f = fed_ols(1), fed_ols(0)

# Propensity model: federated Newton-Raphson. Each round, every client sends its
# gradient and Hessian at the current coefficients; the server sums and updates.
beta, rounds = np.zeros(2), 0
for rounds in range(1, 51):
    g, H = np.zeros(2), np.zeros((2, 2))
    for X, A, Y in clients:                      # runs client-side
        D = design(X)
        p = 1 / (1 + np.exp(-(D @ beta)))
        g += D.T @ (A - p)
        H += D.T @ (D * (p * (1 - p))[:, None])
    step = np.linalg.solve(H, g)                 # runs server-side
    beta += step
    if np.abs(step).max() < 1e-10:
        break

stats = []                                       # (n, sum s, sum s^2) per client
for X, A, Y in clients:                          # runs client-side
    e = clipped(1 / (1 + np.exp(-(design(X) @ beta))))
    s = aipw_scores(X, A, Y, e, b1_f, b0_f)
    stats.append((len(s), s.sum(), (s ** 2).sum()))
N = sum(t[0] for t in stats)
ate_f = sum(t[1] for t in stats) / N
se_f = np.sqrt((sum(t[2] for t in stats) / N - ate_f ** 2) / (N - 1))

# ---- 3. Local-only, combined by inverse variance --------------------------
local = []
for X, A, Y in clients:
    e = clipped(LogisticRegression(C=1e6, max_iter=1000).fit(X.reshape(-1, 1), A)
                .predict_proba(X.reshape(-1, 1))[:, 1])
    b1 = np.linalg.lstsq(design(X[A == 1]), Y[A == 1], rcond=None)[0]
    b0 = np.linalg.lstsq(design(X[A == 0]), Y[A == 0], rcond=None)[0]
    s = aipw_scores(X, A, Y, e, b1, b0)
    local.append((s.mean(), s.std(ddof=1) / np.sqrt(len(s)), e.min(), e.max()))
w = np.array([1 / t[1] ** 2 for t in local])
ate_l = np.sum(w * np.array([t[0] for t in local])) / w.sum()
se_l = 1 / np.sqrt(w.sum())

print("Per-client, local-only AIPW (true ATE = 1.5):")
for k, (est, se, lo, hi) in enumerate(local, 1):
    print(f"  client {k} (n={len(clients[k - 1][0]):>4}): ATE {est:6.3f} (SE {se:.3f})   "
          f"propensity range [{lo:.3f}, {hi:.3f}]")
print()
print(f"{'method':<34}{'ATE':>8}{'SE':>8}")
print(f"{'centralized (rows pooled)':<34}{ate_c:8.3f}{se_c:8.3f}")
print(f"{'federated-exact (aggregates only)':<34}{ate_f:8.3f}{se_f:8.3f}")
print(f"{'local-only, inverse-variance':<34}{ate_l:8.3f}{se_l:8.3f}")
print()
print(f"federated vs centralized |difference|: {abs(ate_f - ate_c):.2e}  "
      f"(Newton rounds used: {rounds})")

Per-client, local-only AIPW (true ATE = 1.5):
  client 1 (n=2000): ATE  1.514 (SE 0.050)   propensity range [0.068, 0.948]
  client 2 (n=1200): ATE  1.529 (SE 0.065)   propensity range [0.135, 0.964]
  client 3 (n= 300): ATE  1.360 (SE 0.340)   propensity range [0.010, 0.990]

method                                 ATE      SE
centralized (rows pooled)            1.519   0.040
federated-exact (aggregates only)    1.519   0.040
local-only, inverse-variance         1.517   0.039

federated vs centralized |difference|: 1.55e-06  (Newton rounds used: 6)


### What breaks when data can't move
- **Exactness depends on the nuisance models.** OLS needs one round of aggregates and
  logistic regression a handful of Newton rounds, and both match the centralized fit.
  The repo's LightGBM / CatBoost learners have no exact federated equivalent, so DR with
  flexible learners would be approximate.
- **Cross-fitting multiplies communication.** Each fold needs its own federated nuisance
  fit, and fold assignment must be coordinated.
- **Positivity is only visible per client unless you share summaries.** In the table above
  the propensity range shows the small, strongly confounded client is close to violating
  positivity locally, which you cannot see from the pooled estimate alone.
- **Local-only plus combining answers a different question when effects differ across
  clients**, and a small client's weight comes only from its own variance estimate.
  (Here the true effect is the same everywhere, which is the easy case.)
- **Aggregates are not automatically private.** `(n, sum, sum of squares)` from a tiny
  client reveals more than from a large one; secure aggregation or differential privacy
  is outside this demo.

## 6. Target trial emulation (TTE): protocol checklist
TTE is a *design* framework, not an estimator: specify the randomized trial you would
ideally run, then emulate each element with observational data (Hernán & Robins, 2016).
The checklist below is a template, not a computation. The grouping into workflow,
protocol elements and pitfalls is our own summary, not a standard.

In [7]:
tte_checklist = {
    "Workflow (our summary)": [
        "Formulate the causal question (PICO)",
        "Specify the target trial protocol",
        "Emulate each protocol element with the observational data",
    ],
    "Protocol elements (target trial)": [
        "Eligibility criteria",
        "Treatment strategies",
        "Assignment procedures",
        "Start and end of follow-up (time zero)",
        "Outcome",
        "Causal contrast (ITT vs per-protocol)",
        "Analysis plan",
    ],
    "Common design pitfalls (non-exhaustive)": [
        "Immortal time bias (align eligibility, assignment and time zero)",
        "Prevalent-user bias (use a new-user design)",
        "Confounding, including time-varying confounding",
        "Selection bias from loss to follow-up or censoring",
    ],
}
for section, items in tte_checklist.items():
    print(f"\n{section}:")
    for i, item in enumerate(items, 1):
        print(f"  [ ] {i}. {item}")


Workflow (our summary):
  [ ] 1. Formulate the causal question (PICO)
  [ ] 2. Specify the target trial protocol
  [ ] 3. Emulate each protocol element with the observational data

Protocol elements (target trial):
  [ ] 1. Eligibility criteria
  [ ] 2. Treatment strategies
  [ ] 3. Assignment procedures
  [ ] 4. Start and end of follow-up (time zero)
  [ ] 5. Outcome
  [ ] 6. Causal contrast (ITT vs per-protocol)
  [ ] 7. Analysis plan

Common design pitfalls (non-exhaustive):
  [ ] 1. Immortal time bias (align eligibility, assignment and time zero)
  [ ] 2. Prevalent-user bias (use a new-user design)
  [ ] 3. Confounding, including time-varying confounding
  [ ] 4. Selection bias from loss to follow-up or censoring


## What this notebook shows
- A naive estimate is biased by confounding (section 1).
- AIPW recovers the true ATE in this simulation, in a single draw and on average
  over 200 replications (sections 2 and 4).
- Cross-fitting works with a flexible learner (section 3). One draw shows it runs;
  it is not a proof that it is better than not cross-fitting.
- Doubly robust estimation can be done from aggregates only, for parametric nuisance
  models, and gives the centralized answer (section 5).
- TTE is a design checklist applied *before* estimation (section 6).

## What it does not show
- Real clinical data, or any claim about NHANES
- Federated estimation with flexible (tree-based) learners
- Sensitivity analysis for unmeasured confounding
- Transportability (see the companion notebook)